# Quality Assessment Orderlines vs Orders


In [2]:
import pandas as pd

Load my cleaned DataFrames

In [3]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Project Eniac/Data/Data_clean/"

orders_cl = pd.read_csv(path + "orders_clean.csv")
orderlines_cl = pd.read_csv(path + "orders_line_clean.csv")
product_cl = pd.read_csv(path + "products_clean.csv")


Mounted at /content/drive


## 1.&nbsp; Define Pandas display format

In [4]:
pd.set_option("display.max_columns", None)

## 2.&nbsp; Exclude unwanted orders

In [5]:
orders_cl["state"].value_counts()

,count
state,
Shopping Basket,117809
Completed,46605
Place Order,40883
Pending,14374
Cancelled,7233


In [6]:
orders_quality_controlled = orders_cl[orders_cl["state"] == "Completed"]

In [7]:
orders_quality_controlled["state"].value_counts()

,count
state,
Completed,46605


## 3.&nbsp; Exclude orders with unknown products


In [8]:
orderlines_cl.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38


In [9]:
product_cl.head()

,sku,name,desc,price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364


In [10]:
orderlines_quality_controlled = orderlines_cl[
    orderlines_cl["sku"].isin(product_cl["sku"])
].copy()

In [11]:
orderlines_cl.shape


(216250, 7)

In [12]:
orderlines_quality_controlled.shape

(212546, 7)

## 4.&nbsp; Explore the revenue from different tables

#### Step 1:
Create the `unit_price_total` as `orderlines.unit_price` * `orderlines.product_quantity`

In [13]:
orderlines_quality_controlled["unit_price_total"] = (orderlines_quality_controlled["unit_price"] * orderlines_quality_controlled["product_quantity"])

In [14]:
orderlines_quality_controlled["unit_price_total"]

,unit_price_total
0,18.99
1,399.00
2,474.05
3,68.39
4,23.74
...,...
216245,42.99
216246,141.58
216247,19.98
216248,19.99


#### Step 2:
Group by `id_order`, summarising by the sum of `unit_price_total`

In [15]:
orderlines_grouped = orderlines_quality_controlled.groupby("id_order")["unit_price_total"].sum()

In [16]:
orderlines_grouped = orderlines_grouped.to_frame()

In [17]:
orderlines_grouped.info()

<class 'pandas.core.frame.DataFrame'>
Index: 167985 entries, 241319 to 527401
Data columns (total 1 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   unit_price_total  167985 non-null  float64
dtypes: float64(1)
memory usage: 2.6 MB


In [18]:
orderlines_grouped = orderlines_grouped.reset_index()

In [19]:
orderlines_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167985 entries, 0 to 167984
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id_order          167985 non-null  int64  
 1   unit_price_total  167985 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 2.6 MB


In [20]:
orders_merged = orders_quality_controlled.merge(orderlines_grouped, left_on='order_id', right_on='id_order', how='left')

In [21]:
orders_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46605 entries, 0 to 46604
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          46605 non-null  int64  
 1   created_date      46605 non-null  object 
 2   total_paid        46605 non-null  float64
 3   state             46605 non-null  object 
 4   id_order          42558 non-null  float64
 5   unit_price_total  42558 non-null  float64
dtypes: float64(3), int64(1), object(2)
memory usage: 2.1+ MB


In [22]:
orders_merged[orders_merged["unit_price_total"].isna()]

,order_id,created_date,total_paid,state,id_order,unit_price_total
5,245941,2017-01-01 10:32:23,183.52,Completed,NaN,NaN
25,257847,2017-11-23 23:46:54,1367.11,Completed,NaN,NaN
28,258985,2017-07-31 12:52:38,2264.60,Completed,NaN,NaN
31,259668,2017-10-06 22:06:58,1132.33,Completed,NaN,NaN
36,262016,2017-08-18 01:05:38,3109.57,Completed,NaN,NaN
...,...,...,...,...,...,...
46501,526363,2018-03-13 11:47:03,166.98,Completed,NaN,NaN
46506,526380,2018-03-13 12:16:06,1436.99,Completed,NaN,NaN
46532,526505,2018-03-13 15:56:14,1421.99,Completed,NaN,NaN
46533,526507,2018-03-13 16:04:20,1137.97,Completed,NaN,NaN


This is likely because all order lines for these orders were removed during the data cleaning process.

### What is the average difference between `total_paid` and `unit_price_total`?

In [23]:
orders_merged['difference'] = orders_merged['total_paid'] - orders_merged['unit_price_total']

In [24]:
orders_merged['difference'].mean()

np.float64(5.718680624089478)

The average difference between total_paid and unit_price_total is approximately €5.72. However, the average alone is not sufficient to explain the differences, so the distribution should be analyzed next.

### What is the distribution of these differences?

In [25]:
orders_merged["difference"] = orders_merged["difference"].round(2)

In [26]:
# your code here
orders_merged['difference'].value_counts()

,count
difference,
0.00,11202
4.99,11062
6.99,10012
3.99,6861
19.99,546
...,...
312.97,1
24.27,1
303.58,1


### Next step

In [27]:
orders_final = orders_merged[
    orders_merged["difference"].notna()
].copy()

Orders with unexplained differences should be investigated. If the differences cannot be explained, these orders should be excluded from the final analysis because they may contain data quality issues.

## 5.&nbsp; Confident about my dataset

The dataset was cleaned by keeping only completed orders and removing order lines with unknown products. Revenue analysis showed that most differences between total_paid and unit_price_total are fixed amounts (€0.00, €3.99, €4.99, and €6.99), which are likely related to shipping costs. A small number of orders have unusually large differences that cannot be explained by shipping alone. These orders should be investigated further and excluded from the final analysis if no explanation is found.

In [28]:
path = "/content/drive/MyDrive/Project Eniac/Data/Data_clean/"

orders_final.to_csv(path + "orders_qu.csv", index=False)
orderlines_quality_controlled.to_csv(path + "orderlines_qu.csv", index=False)